In [ ]:
!pip install --upgrade biopython wordcloud transformers tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 28.1 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
  Attempting uninstall: tf-keras
    Found existing installation: tf_keras 2.19.0
    Uninstalling tf_keras-2.19.0:
      Successfully uninstalled tf_keras-2.19.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstal

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.41.2 tf-keras

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gc
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from google.colab import drive

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from Bio import Entrez
from transformers import AutoTokenizer, TFAutoModel

In [ ]:
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/NLP_Project_Final'
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
def reset_env(seed=42):
    tf.keras.backend.clear_session()
    gc.collect()
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(">>> Environment cleaned and Seed fixed.")

reset_env()

>>> Environment cleaned and Seed fixed.


# **DATASET AND PREPROCESSING**

## **Scrapping data**

In [ ]:
Entrez.email = "22011108@st.phenikaa-uni.edu.vn"

METHODOLOGY_QUERIES = {
    "Randomized Controlled Trial": '"Randomized Controlled Trial"[Publication Type]',
    "Case Report": '"Case Reports"[Publication Type]',
    "Meta-Analysis": '"Meta-Analysis"[Publication Type]',
    "Observational Study": '"Observational Study"[Publication Type]'
}

In [ ]:
def fetch_and_save_csv(samples_per_class=10000):
    dataset = []
    print(">>> [PROCESS] Starting PubMed Data Acquisition Strategy...")

    for label, query in METHODOLOGY_QUERIES.items():
        print(f"[*] Fetching Category: {label}...")
        try:
            search_query = f"{query} AND hasabstract[text] AND English[Language]"

            handle = Entrez.esearch(db="pubmed", term=search_query, retmax=samples_per_class)
            id_list = Entrez.read(handle)["IdList"]
            handle.close()

            print(f"    - Found {len(id_list)} IDs. Downloading in batches...")

            for i in range(0, len(id_list), 200):
                batch_ids = id_list[i : i + 200]
                handle = Entrez.efetch(db="pubmed", id=batch_ids, retmode="xml")
                records = Entrez.read(handle)
                handle.close()

                for article in records.get('PubmedArticle', []):
                    try:
                        abstract_parts = article['MedlineCitation']['Article']['Abstract']['AbstractText']
                        full_abstract = " ".join([str(t) for t in abstract_parts])
                        dataset.append({"text": full_abstract, "methodology": label})
                    except (KeyError, TypeError):
                        continue

                time.sleep(0.3)

        except Exception as e:
            print(f"    [!] Error during {label} retrieval: {e}")
            continue

    df = pd.DataFrame(dataset)

    csv_filename = 'pubmed_dataset_final.csv'
    csv_path = os.path.join(SAVE_DIR, csv_filename)

    df.to_csv(csv_path, index=False, encoding='utf-8-sig')

    print("-" * 50)
    print(f"[SUCCESS] Total samples collected: {len(df)}")
    print(f"[ARCHIVED] Dataset successfully secured at Drive: {csv_path}")
    print("-" * 50)

    return df

df_raw = fetch_and_save_csv(samples_per_class=10000)

>>> [PROCESS] Starting PubMed Data Acquisition Strategy...
[*] Fetching Category: Randomized Controlled Trial...
    - Found 9999 IDs. Downloading in batches...
[*] Fetching Category: Case Report...
    - Found 9999 IDs. Downloading in batches...
[*] Fetching Category: Meta-Analysis...
    - Found 9999 IDs. Downloading in batches...
[*] Fetching Category: Observational Study...
    - Found 9999 IDs. Downloading in batches...
--------------------------------------------------
[SUCCESS] Total samples collected: 39996
[ARCHIVED] Dataset successfully secured at Drive: /content/drive/MyDrive/NLP_Project_Final/pubmed_dataset_final.csv
--------------------------------------------------
